# Phase 10.4 — Deterministic Baseline Ranker

Weighted scoring over **normalized** features (Phase 10.3) → `RankingResponse`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.ranking.adapters import ranking_candidate_from_retrieval
from productiq.ranking.baseline_config import BaselineRankingConfig
from productiq.ranking.baseline_ranker import rank_candidates
from productiq.ranking.baseline_scoring import compute_baseline_score
from productiq.ranking.contracts import RankingRequest
from productiq.ranking.feature_schema import RankingFeatures
from productiq.ranking.normalization import normalize_feature_rows
from productiq.representation.query_contract import build_query_representation
from productiq.retrieval.contracts import RetrievalCandidate, RetrievalMethod

In [ ]:
query = build_query_representation("nike running shoes")
raw_rows = (
    RankingFeatures(
        product_id="P_low",
        retrieval={
            "bm25_score": 1.0,
            "rrf_score": 0.2,
            "retrieved_by_bm25": True,
            "retrieved_by_vector": False,
            "retrieved_by_both": False,
        },
        matching={"matched_attribute_count": 0, "brand_match": False},
        catalog={},
    ),
    RankingFeatures(
        product_id="P_high",
        retrieval={
            "bm25_score": 5.0,
            "rrf_score": 0.9,
            "retrieved_by_bm25": True,
            "retrieved_by_vector": True,
            "retrieved_by_both": True,
        },
        matching={"matched_attribute_count": 2, "brand_match": True, "attribute_overlap": 0.8},
        catalog={"discount_price_inr": 999, "original_price_inr": 1999, "discount_amount_inr": 1000},
    ),
    RankingFeatures(
        product_id="P_tie_a",
        retrieval={
            "bm25_score": 3.0,
            "rrf_score": 0.5,
            "retrieved_by_bm25": True,
            "retrieved_by_vector": False,
            "retrieved_by_both": False,
        },
        matching={"matched_attribute_count": 1, "brand_match": True},
        catalog={},
    ),
    RankingFeatures(
        product_id="P_tie_b",
        retrieval={
            "bm25_score": 4.0,
            "rrf_score": 0.5,
            "retrieved_by_bm25": True,
            "retrieved_by_vector": False,
            "retrieved_by_both": False,
        },
        matching={"matched_attribute_count": 1, "brand_match": True},
        catalog={},
    ),
)
normalized = normalize_feature_rows(raw_rows)
feature_map = {row.product_id: row for row in normalized}
candidates = tuple(
    ranking_candidate_from_retrieval(
        RetrievalCandidate(product_id=row.product_id, score=1.0, method=RetrievalMethod.BM25)
    )
    for row in raw_rows
)
request = RankingRequest(query=query, candidates=candidates, top_k=3)
config = BaselineRankingConfig()
response = rank_candidates(
    request=request,
    normalized_features_by_product_id=feature_map,
    baseline_config=config,
)
response.model_dump(mode="json")

In [ ]:
breakdown = compute_baseline_score(feature_map["P_high"], config=config)
{
    "product_id": breakdown.product_id,
    "ranking_score": breakdown.ranking_score,
    "nonzero_contributions": [(n, c) for n, c in breakdown.contributions if c],
}

Notes:
- Catalog fields are normalized in 10.3 but **excluded** from baseline weights (0.0).
- Tie scores use `product_id` ordering (`P_tie_a` before `P_tie_b`).
- `top_k=3` returns three ranked rows; full candidate pool had four items.